# Technical Note P4: NumPy Vector, Matrix, Tensor Operations

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{shape}
\rightarrow
\text{axis}
\rightarrow
\text{slicing / broadcasting}
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. Four different products

Separate elementwise multiplication, inner products, matrix multiplication, and outer products. Similar syntax or output shapes do not make these operations interchangeable.

| Operation | NumPy | Meaning |  
| --- | --- | --- |  
| elementwise multiplication | `a * b` | Multiply corresponding entries |  
| inner product | `a @ b`, `np.dot(a,b)` | component contraction |  
| matrix-vector multiplication | `A @ x` | linear map |  
| outer product | `np.outer(a,b)` | Form all component pairs |

$$
\boxed{\texttt{*}\neq\texttt{@}}
$$

In [2]:
import numpy as np

a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

print("a * b =", a * b)
print("a @ b =", a @ b)
print("np.dot(a,b) =", np.dot(a, b))

a * b = [ 4. 10. 18.]
a @ b = 32.0
np.dot(a,b) = 32.0


## 2. Elementwise multiplication

Multiply corresponding entries with star. This is a componentwise relationship, not a general linear map or invariant tensor operation.

$$
\mathbf a=
\begin{bmatrix}
a_1\\a_2\\a_3
\end{bmatrix},
\qquad
\mathbf b=
\begin{bmatrix}
b_1\\b_2\\b_3
\end{bmatrix}
$$

```python
a * b
```

$$
\begin{bmatrix}
a_1b_1\\a_2b_2\\a_3b_3
\end{bmatrix}
$$

In [3]:
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

c = a * b
print(c)
print("shape =", c.shape)

[ 4. 10. 18.]
shape = (3,)


## 3. Inner product

Contract matching vector components to obtain a scalar. For complex vectors use the intended conjugation convention; real examples here use ordinary Euclidean products.

$$
\mathbf a\cdot\mathbf b=
\sum_i a_i b_i
$$

In [4]:
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

print(a @ b)
print(np.dot(a, b))
print(np.sum(a*b))

32.0
32.0
32.0


$$
\boxed{
\mathbf a\cdot\mathbf b
\rightarrow
\text{scalar}
}
$$

## 4. Geometric meaning of an inner product

The inner product measures alignment and determines the cosine of the angle between nonzero vectors. It depends on the metric or orthonormal coordinate convention.

$$
\mathbf a\cdot\mathbf b=
\|\mathbf a\|\,\|\mathbf b\|\cos\theta
$$

In [5]:
a = np.array([1., 0.])
b = np.array([1., 1.])

cos_theta = (a @ b) / (np.linalg.norm(a)*np.linalg.norm(b))
theta = np.degrees(np.arccos(cos_theta))

print("cos(theta) =", cos_theta)
print("theta [deg] =", theta)

cos(theta) = 0.7071067811865475
theta [deg] = 45.00000000000001


## 5. Norm

A norm measures magnitude. State the norm and units used when judging errors or residuals.

$$
\|\mathbf a\|=
\sqrt{\mathbf a\cdot\mathbf a}
$$

In [6]:
a = np.array([3., 4.])

print(np.linalg.norm(a))
print(np.sqrt(a @ a))

5.0
5.0


## 6. Unit vectors

Normalize only a nonzero vector. A zero or very small norm needs an explicit handling rule rather than an arbitrary division.

$$
\hat{\mathbf a}=
\frac{\mathbf a}{\|\mathbf a\|}
$$

In [7]:
a = np.array([3., 4.])
ahat = a / np.linalg.norm(a)

print("ahat =", ahat)
print("norm =", np.linalg.norm(ahat))

ahat = [0.6 0.8]
norm = 1.0


## 7. Scalar projection

The projection onto a unit direction is a signed scalar. Multiplying that scalar by the direction gives the vector projection.

$$
v_n=\mathbf n\cdot\mathbf v
$$

In [8]:
v = np.array([3., 2.])
n = np.array([1., 1.])
n = n / np.linalg.norm(n)

vn = n @ v

print("scalar projection =", vn)

scalar projection = 3.5355339059327373


$$
\mathbf v_n=
(\mathbf n\cdot\mathbf v)\mathbf n
$$

In [9]:
v_proj = (n @ v) * n
print(v_proj)

[2.5 2.5]


## 8. Orthogonal decomposition

Split a vector into its projection and the remaining perpendicular part. Check reconstruction and orthogonality.

$$
\mathbf v=
(\mathbf n\cdot\mathbf v)\mathbf n
+
(\mathbf m\cdot\mathbf v)\mathbf m
$$

In [10]:
theta = np.deg2rad(30)

n = np.array([np.cos(theta), np.sin(theta)])
m = np.array([-np.sin(theta), np.cos(theta)])

v = np.array([4., 1.])

vn = n @ v
vm = m @ v
v_rec = vn*n + vm*m

print("n·m =", n @ m)
print("reconstructed =", v_rec)
print("same?", scaled_allclose(v, v_rec))

n·m = 7.437084071668725e-18
reconstructed = [4. 1.]
same? True


## 9. Outer product

The outer product creates a second-order array from two vectors. Its action on another vector provides a useful interpretation as a linear map.

$$
(\mathbf a\otimes\mathbf b)_{ij}=
a_i b_j
$$

In [11]:
a = np.array([1., 2.])
b = np.array([3., 4., 5.])

C = np.outer(a, b)

print(C)
print("shape =", C.shape)

[[ 3.  4.  5.]
 [ 6.  8. 10.]]
shape = (2, 3)


$$
\boxed{
\mathbf a\cdot\mathbf b\rightarrow\text{scalar}
}
$$

$$
\boxed{
\mathbf a\otimes\mathbf b\rightarrow\text{matrix}
}
$$

## 10. Matrix-vector product

The matrix maps input components to output components. Its column count matches the input dimension and its row count determines the output dimension.

$$
\mathbf y=\mathbf A\mathbf x
$$

$$
y_i=\sum_jA_{ij}x_j
$$

In [12]:
A = np.array([
    [2., 1.],
    [1., 3.],
])

x = np.array([4., 5.])

y = A @ x
print(y)

[13. 19.]


## 11. A matrix as a linear map

Interpret columns as the outputs produced by input basis vectors. This connects component arithmetic to the structure of the transformation.

$$
\boxed{
\mathbf x
\mapsto
\mathbf A\mathbf x
}
$$

In [13]:
A = np.array([
    [2., 0.],
    [0., 0.5],
])

x = np.array([1., 2.])

print("input =", x)
print("output =", A @ x)

input = [1. 2.]
output = [2. 1.]


## 12. Matrix-matrix product

Multiplication composes linear maps in right-to-left order. The contracted dimensions must match.

$$
\mathbf C=\mathbf A\mathbf B
$$

$$
C_{ik}=\sum_jA_{ij}B_{jk}
$$

In [14]:
A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[2., 0.],
              [1., 2.]])

C = A @ B

print(C)

[[ 4.  4.]
 [10.  8.]]


$$
(m,n)(n,p)\rightarrow(m,p)
$$

In [15]:
A = np.zeros((3, 4))
B = np.zeros((4, 2))
C = A @ B

print(A.shape, B.shape, C.shape)

(3, 4) (4, 2) (3, 2)


## 13. Transpose

For a matrix, transpose exchanges its two component axes. It changes input and output roles rather than physically rotating a material.

$$
(A^T)_{ij}=A_{ji}
$$

In [16]:
A = np.array([
    [1., 2., 3.],
    [4., 5., 6.],
])

print(A)
print()
print(A.T)

[[1. 2. 3.]
 [4. 5. 6.]]

[[1. 4.]
 [2. 5.]
 [3. 6.]]


## 14. Transpose of a one-dimensional array

A one-dimensional array has only one axis, so dot-T does not turn it into a column. Introduce a singleton axis explicitly.

In [17]:
v = np.array([1., 2., 3.])

print("v shape =", v.shape)
print("v.T shape =", v.T.shape)

v shape = (3,)
v.T shape = (3,)


```python
v[:,None]
v[None,:]
```

In [18]:
v = np.array([1., 2., 3.])

v_col = v[:, None]
v_row = v[None, :]

print(v_col.shape)
print(v_row.shape)
print(v_col @ v_row)

(3, 1)
(1, 3)
[[1. 2. 3.]
 [2. 4. 6.]
 [3. 6. 9.]]


## 15. Identity

The identity map preserves components. Its size is tied to the space on which it acts.

$$
\mathbf I\mathbf x=\mathbf x
$$

In [19]:
I = np.eye(3)
x = np.array([2., 4., 6.])

print(I @ x)

[2. 4. 6.]


## 16. Diagonal matrices

A diagonal map scales components independently in the chosen basis. It need not remain diagonal after a basis rotation.

$$
\mathbf D=\operatorname{diag}(d_1,d_2,d_3)
$$

In [20]:
d = np.array([1., 2., 4.])
D = np.diag(d)

print(D)

[[1. 0. 0.]
 [0. 2. 0.]
 [0. 0. 4.]]


## 17. Diagonal anisotropic flux

A diagonal diffusivity multiplies each gradient component by its directional coefficient. The basis must be the material principal basis for this interpretation.

$$
\mathbf q=-\mathbf D\nabla u
$$

In [21]:
Dvec = np.array([1.0, 0.5, 2.0])
grad_u = np.array([3.0, 4.0, 5.0])

Dmat = np.diag(Dvec)

q1 = -(Dmat @ grad_u)
q2 = -(Dvec * grad_u)

print(q1)
print(q2)

[ -3.  -2. -10.]
[ -3.  -2. -10.]


## 18. Full tensor flux

A full diffusivity tensor contracts with the gradient. Off-diagonal entries couple the coordinate components.

$$
\mathbf q=-\mathbf D\nabla u
$$

In [22]:
D = np.array([
    [2.0, 0.4],
    [0.4, 1.0],
])

grad_u = np.array([1.5, -0.5])

q = -(D @ grad_u)

print(q)

[-2.8 -0.1]


## 19. Quadratic forms

A quadratic form evaluates a scalar from a vector and a matrix. Positive definiteness concerns its sign for every nonzero vector.

$$
q=\mathbf x^T\mathbf A\mathbf x
$$

In [23]:
A = np.array([
    [2., 0.5],
    [0.5, 1.],
])
x = np.array([1., 2.])

q = x @ A @ x

print(q)

8.0


## 20. Symmetry

A real matrix is symmetric when it equals its transpose. Check with tolerances appropriate to the data scale.

$$
\mathbf A^T=\mathbf A
$$

In [24]:
A = np.array([
    [2., 0.5],
    [0.5, 1.],
])

print(scaled_allclose(A, A.T))

True


## 21. Matrix norms

The default array norm and the induced spectral norm are different quantities. Specify the norm when comparing operators or estimating amplification.

$$
\|\mathbf A\|_F=
\sqrt{\sum_{ij}A_{ij}^2}
$$

In [25]:
A = np.array([[1., 2.],
              [3., 4.]])

print(np.linalg.norm(A))

5.477225575051661


## 22. Trace

Trace is the sum of diagonal entries and is invariant under similarity transformations. A tensor-field trace must act on component axes.

$$
\operatorname{tr}(\mathbf A)=\sum_iA_{ii}
$$

In [26]:
A = np.array([[1., 2.],
              [3., 4.]])

print(np.trace(A))

5.0


## 23. Determinant

The determinant describes oriented volume scaling for a linear map. Its magnitude alone is a poor numerical conditioning diagnostic.

$$
\det\mathbf A
$$

In [27]:
A = np.array([[2., 1.],
              [1., 3.]])

print(np.linalg.det(A))

5.000000000000001


$$
J=\det\mathbf F
$$

## 24. Solve instead of forming an inverse

Use a linear solve for Ax=b. An explicit inverse is generally unnecessary and may cost more while adding numerical error.

$$
\mathbf x=\mathbf A^{-1}\mathbf b
$$

In [28]:
A = np.array([[3., 1.],
              [1., 2.]])
b = np.array([9., 8.])

x_solve = np.linalg.solve(A, b)
x_inv = np.linalg.inv(A) @ b

print("solve =", x_solve)
print("inv @ b =", x_inv)

solve = [2. 3.]
inv @ b = [2. 3.]


## 25. Eigenvalues and eigenvectors

An eigenvector identifies an invariant direction of a map. Use eigh for real symmetric matrices and inspect residuals; eigenvector signs are not unique.

$$
\mathbf A\mathbf v=\lambda\mathbf v
$$

In [29]:
A = np.array([[2., 1.],
              [1., 2.]])

eigvals, eigvecs = np.linalg.eigh(A)

print("eigenvalues =", eigvals)
print("eigenvectors =")
print(eigvecs)

eigenvalues = [1. 3.]
eigenvectors =
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


## 26. Principal stresses

For a symmetric Cauchy stress tensor, eigenvalues are principal stresses and eigenvectors are principal directions. Repeated eigenvalues make directions nonunique.

$$
\boldsymbol\sigma=
\begin{bmatrix}
\sigma_x&\tau_{xy}\\
\tau_{xy}&\sigma_y
\end{bmatrix}
$$

In [30]:
sigma = np.array([
    [10.0, 3.0],
    [3.0, 4.0],
])

principal_stress, principal_dirs = np.linalg.eigh(sigma)

print("principal stresses =", principal_stress)
print("principal directions =")
print(principal_dirs)

principal stresses = [ 2.75735931 11.24264069]
principal directions =
[[ 0.38268343 -0.92387953]
 [-0.92387953 -0.38268343]]


## 27. Rotation matrices

A proper rotation satisfies Q-transpose-Q=I and determinant one. Orthogonality alone also permits reflections.

$$
\mathbf Q=
\begin{bmatrix}
\cos\theta&-\sin\theta\\
\sin\theta&\cos\theta
\end{bmatrix}
$$

In [31]:
theta = np.deg2rad(30)

Q = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)],
])

print(Q)
print("Q.T @ Q =")
print(Q.T @ Q)

[[ 0.8660254 -0.5      ]
 [ 0.5        0.8660254]]
Q.T @ Q =
[[1.00000000e+00 7.43708407e-18]
 [7.43708407e-18 1.00000000e+00]]


$$
\mathbf Q^T\mathbf Q=\mathbf I
$$

## 28. Rotate a vector

Qv can describe active rotation of a vector in a fixed basis. A passive basis change uses a different interpretation of the same algebra.

In [32]:
v = np.array([1., 0.])
v_rot = Q @ v

print(v_rot)

[0.8660254 0.5      ]


## 29. Rotate a second-order tensor

Under the declared orthonormal convention, use QAQ-transpose. Declare whether Q actively rotates the object or supplies local basis vectors in global coordinates.

$$
\mathbf A'=\mathbf Q\mathbf A\mathbf Q^T
$$

In [33]:
A = np.array([
    [2.0, 0.0],
    [0.0, 1.0],
])

A_rot = Q @ A @ Q.T

print(A_rot)

[[1.75      0.4330127]
 [0.4330127 1.25     ]]


## 30. Rotated anisotropic diffusivity

Rotating unequal principal coefficients generally creates off-diagonal entries. These entries encode orientation rather than a new measured transport mechanism.

$$
\mathbf D_{\mathrm{mat}}=
\operatorname{diag}(D_1,D_2)
$$

$$
\boxed{
\mathbf D=
\mathbf Q\mathbf D_{\mathrm{mat}}\mathbf Q^T
}
$$

In [34]:
Dmat = np.diag([2.0, 0.5])
Dglobal = Q @ Dmat @ Q.T

print("D material =")
print(Dmat)
print("\nD global =")
print(Dglobal)

D material =
[[2.  0. ]
 [0.  0.5]]

D global =
[[1.625      0.64951905]
 [0.64951905 0.875     ]]


## 31. Traction

Cauchy traction on a plane with unit normal n is sigma-n. The sign convention for stress and outward normal must be stated.

$$
\boxed{
\mathbf t=\boldsymbol\sigma\mathbf n
}
$$

In [35]:
sigma = np.array([
    [12.0, 4.0],
    [4.0, 6.0],
])

theta = np.deg2rad(35)
n = np.array([np.cos(theta), np.sin(theta)])

t = sigma @ n

print("n =", n)
print("traction t =", t)

n = [0.81915204 0.57357644]
traction t = [12.12413028  6.7180668 ]


$$
\boxed{
\mathbf n\mapsto\mathbf t
}
$$

## 32. Normal and shear stress

Normal traction is the scalar n-dot-t times n; the remainder is shear traction. The shear vector is tangent to the plane.

$$
\mathbf m=
\begin{bmatrix}
-\sin\theta\\
\cos\theta
\end{bmatrix}
$$

$$
\sigma_n=\mathbf n\cdot\mathbf t
$$

$$
\tau_n=\mathbf m\cdot\mathbf t
$$

In [36]:
m = np.array([-np.sin(theta), np.cos(theta)])

sigma_n = n @ t
tau_n = m @ t

print("sigma_n =", sigma_n)
print("tau_n =", tau_n)

sigma_n = 13.784830913120638
tau_n = -1.450997289055049


$$
\boxed{
\boldsymbol\sigma
\xrightarrow{\ \mathbf n\ }
\mathbf t
\xrightarrow{\ \mathbf n,\mathbf m\ }
(\sigma_n,\tau_n)
}
$$

## 33. Reconstruct traction

Add the normal and shear vectors and check tangency. These component identities provide a simple mechanical verification.

$$
\mathbf t=
\sigma_n\mathbf n+\tau_n\mathbf m
$$

In [37]:
t_rec = sigma_n*n + tau_n*m

print("t =", t)
print("reconstructed =", t_rec)
print("same?", scaled_allclose(t, t_rec))

t = [12.12413028  6.7180668 ]
reconstructed = [12.12413028  6.7180668 ]
same? True


## 34. Cross product

In three-dimensional Euclidean space the cross product gives an oriented perpendicular vector. Axis orientation determines its sign.

$$
\mathbf c=\mathbf a\times\mathbf b
$$

In [38]:
a = np.array([1., 0., 0.])
b = np.array([0., 1., 0.])

print(np.cross(a, b))

[0. 0. 1.]


## 35. Projectors from outer products

For unit n, the normal projector is n-outer-n and the tangent projector is I minus that projector. Normalization is essential.

$$
\mathbf P_n=\mathbf n\otimes\mathbf n
$$

In [39]:
n = np.array([1., 2.])
n = n / np.linalg.norm(n)

P = np.outer(n, n)

print(P)
print("P @ n =", P @ n)

[[0.2 0.4]
 [0.4 0.8]]
P @ n = [0.4472136  0.89442719]


$$
\mathbf P_t=\mathbf I-\mathbf n\otimes\mathbf n
$$

In [40]:
Pt = np.eye(2) - P
v = np.array([3., 1.])

vn = P @ v
vt = Pt @ v

print("normal =", vn)
print("tangent =", vt)
print("sum =", vn + vt)

normal = [1. 2.]
tangent = [ 2. -1.]
sum = [3. 1.]


## 36. Tensor contraction

A contraction sums paired component indices. Write the index expression before implementing a multidimensional array operation.

$$
y_i=A_{ij}x_j
$$

$$
c=A_{ij}B_{ij}
$$

$$
C_{ik}=A_{ij}B_{jk}
$$

## 37. Double contraction

For real second-order arrays, A:B sums Aij-Bij. It produces a scalar and differs from an ordinary matrix product.

$$
\mathbf A:\mathbf B=
\sum_{ij}A_{ij}B_{ij}
$$

In [41]:
A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[5., 6.],
              [7., 8.]])

c1 = np.sum(A * B)
c2 = np.einsum("ij,ij->", A, B)

print(c1, c2)

70.0 70.0


$$
\boldsymbol\sigma:\boldsymbol\varepsilon
$$

## 38. Strain energy density

For small-strain linear elasticity, energy is one half epsilon:C:epsilon. Use compatible stress-strain measures and actual tensor shear components.

$$
W=\frac12\boldsymbol\sigma:\boldsymbol\varepsilon
$$

In [42]:
sigma = np.array([
    [10., 2.],
    [2., 5.],
])

eps = np.array([
    [0.01, 0.002],
    [0.002, -0.003],
])

W = 0.5 * np.sum(sigma * eps)

print("W =", W)

W = 0.04650000000000001


## 39. Tensordot

Tensordot contracts explicitly selected axes. Track the order of the uncontracted output axes.

In [43]:
A = np.arange(6).reshape(2, 3)
x = np.array([1., 2., 3.])

y = np.tensordot(A, x, axes=([1], [0]))

print(y)
print("same as A@x?", scaled_allclose(y, A @ x))

[ 8. 26.]
same as A@x? True


## 40. Einsum

Einsum makes contracted and retained indices visible. An explicit output specification avoids relying on implicit axis ordering.

$$
y_i=A_{ij}x_j
$$

In [44]:
A = np.array([[2., 1.],
              [3., 4.]])
x = np.array([5., 6.])

y = np.einsum("ij,j->i", A, x)

print(y)

[16. 39.]


$$
C_{ik}=A_{ij}B_{jk}
$$

In [45]:
B = np.array([[1., 0.],
              [2., 1.]])

C = np.einsum("ij,jk->ik", A, B)

print(C)
print("same as @ ?", scaled_allclose(C, A @ B))

[[ 4.  1.]
 [11.  4.]]
same as @ ? True


## 41. Traction with einsum

Express sigma-ij-n-j to retain traction index i. The formula makes the physical contraction clear.

$$
t_i=\sigma_{ij}n_j
$$

In [46]:
sigma = np.array([
    [12., 4.],
    [4., 6.],
])
n = np.array([0.8, 0.6])

t = np.einsum("ij,j->i", sigma, n)

print(t)

[12.   6.8]


## 42. Tensor field acting on vector field

Use ellipsis for leading location axes and contract the last component axis. Declare the field locations and shapes.

$$
q_i(x,y)=
-D_{ij}(x,y)g_j(x,y)
$$

```python
D.shape = (Nx,Ny,2,2)
g.shape = (Nx,Ny,2)
```

In [47]:
Nx, Ny = 4, 5

D = np.zeros((Nx, Ny, 2, 2))
D[..., 0, 0] = 2.0
D[..., 1, 1] = 1.0
D[..., 0, 1] = 0.3
D[..., 1, 0] = 0.3

g = np.ones((Nx, Ny, 2))

q = -np.einsum("...ij,...j->...i", D, g)

print("D shape =", D.shape)
print("g shape =", g.shape)
print("q shape =", q.shape)
print("q[0,0] =", q[0,0])

D shape = (4, 5, 2, 2)
g shape = (4, 5, 2)
q shape = (4, 5, 2)
q[0,0] = [-2.3 -1.3]


```python
D.shape = (Nx,Ny,Nz,3,3)
g.shape = (Nx,Ny,Nz,3)
```

```python
q = -np.einsum("...ij,...j->...i", D, g)
```

## 43. Batched matrix products

The at operator contracts the last matrix axes and broadcasts leading batch axes. Batched vectors may need a singleton final axis.

In [48]:
A = np.zeros((4, 5, 2, 2))
A[..., 0, 0] = 2.0
A[..., 1, 1] = 3.0

x = np.ones((4, 5, 2, 1))

y = A @ x

print("A shape =", A.shape)
print("x shape =", x.shape)
print("y shape =", y.shape)

A shape = (4, 5, 2, 2)
x shape = (4, 5, 2, 1)
y shape = (4, 5, 2, 1)


## 44. Vector-field norms

Reduce the component axis while retaining spatial axes. A norm over the whole ndarray instead mixes all locations.

```python
V.shape == (...,3)
```

In [49]:
V = np.array([
    [3., 4., 0.],
    [1., 2., 2.],
])

mag = np.linalg.norm(V, axis=-1)

print(mag)

[5. 3.]


## 45. Tensor-field trace

Trace the two component axes explicitly. Default axes may be spatial axes in a high-dimensional array.

In [50]:
T = np.zeros((4, 5, 3, 3))
T[..., 0, 0] = 1.0
T[..., 1, 1] = 2.0
T[..., 2, 2] = 3.0

trT = np.trace(T, axis1=-2, axis2=-1)

print("shape =", trT.shape)
print("unique =", np.unique(trT))

shape = (4, 5)
unique = [6.]


## 46. Transpose tensor components

Swap only the last two axes to transpose each tensor. Dot-T reverses every axis of a multidimensional array.

In [51]:
Tt = np.swapaxes(T, -1, -2)

print("T shape  =", T.shape)
print("Tt shape =", Tt.shape)

T shape  = (4, 5, 3, 3)
Tt shape = (4, 5, 3, 3)


In [52]:
A = np.zeros((4, 5, 3, 3))

print("A shape   =", A.shape)
print("A.T shape =", A.T.shape)
print("swapaxes  =", np.swapaxes(A, -1, -2).shape)

A shape   = (4, 5, 3, 3)
A.T shape = (3, 3, 5, 4)
swapaxes  = (4, 5, 3, 3)


## 47. Orthonormal basis matrices

Put local basis vectors in the columns of Q, expressed in global coordinates. Orthogonality allows Q-transpose to invert the component transformation.

$$
\mathbf Q=
[\mathbf e_1\ \mathbf e_2\ \mathbf e_3]
$$

$$
\mathbf Q^T\mathbf Q=\mathbf I
$$

In [53]:
e1 = np.array([1., 0., 0.])
e2 = np.array([0., 1., 0.])
e3 = np.array([0., 0., 1.])

Q3 = np.column_stack([e1, e2, e3])

print(Q3)
print(Q3.T @ Q3)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 48. Coordinate transformation

With this basis convention, global components equal Q times local components and local components equal Q-transpose times global components. The physical vector is unchanged.

$$
\mathbf v_{\mathrm{global}}=
\mathbf Q\mathbf v_{\mathrm{local}}
$$

$$
\mathbf v_{\mathrm{local}}=
\mathbf Q^T\mathbf v_{\mathrm{global}}
$$

In [54]:
theta = np.deg2rad(30)

eR = np.array([np.cos(theta), np.sin(theta)])
eT = np.array([-np.sin(theta), np.cos(theta)])

Q = np.column_stack([eR, eT])

v_local = np.array([2.0, 1.0])
v_global = Q @ v_local
v_local_back = Q.T @ v_global

print("Q =")
print(Q)
print("v_global =", v_global)
print("v_local_back =", v_local_back)

Q =
[[ 0.8660254 -0.5      ]
 [ 0.5        0.8660254]]
v_global = [1.23205081 1.8660254 ]
v_local_back = [2. 1.]


## 49. Transform a diffusivity tensor

Use Dglobal=Q-Dlocal-Q-transpose for an orthonormal local frame. Verify both quadratic-form and flux invariance.

$$
\mathbf D_{\mathrm{mat}}=
\operatorname{diag}(D_R,D_T)
$$

$$
\mathbf D_g=
\mathbf Q\mathbf D_{\mathrm{mat}}\mathbf Q^T
$$

In [55]:
DR = 2.0
DT = 0.5

Dmat = np.diag([DR, DT])
Dg = Q @ Dmat @ Q.T

print(Dg)

[[1.625      0.64951905]
 [0.64951905 0.875     ]]


## 50. Positive definiteness

For a real symmetric tensor, all eigenvalues must be positive for strict positive definiteness. Testing diagonal entries alone is insufficient.

In [56]:
eigvals = np.linalg.eigvalsh(Dg)

print("eigenvalues =", eigvals)
print("positive definite?", np.all(eigvals > 0))

eigenvalues = [0.5 2. ]
positive definite? True


## 51. Condition number

The condition number measures potential sensitivity in a specified norm. Its interpretation depends on scaling and on the problem being solved.

$$
\kappa(\mathbf A)
$$

In [57]:
A = np.array([
    [1.0, 0.999999],
    [0.999999, 0.9999981],
])

print("condition number =", np.linalg.cond(A))

condition number = 40000322.05362146


## 52. Singular value decomposition

SVD factors a linear map into orthogonal directions and singular scales. It provides a basis for rank and conditioning analysis.

$$
\mathbf A=\mathbf U\boldsymbol\Sigma\mathbf V^T
$$

In [58]:
A = np.array([
    [1., 2.],
    [3., 4.],
    [5., 6.],
])

U, s, VT = np.linalg.svd(A, full_matrices=False)

print("singular values =", s)
print("reconstruction error =", np.linalg.norm(A - U @ np.diag(s) @ VT))

singular values = [9.52551809 0.51430058]
reconstruction error = 1.2755491433176288e-15


## 53. Numerical rank

Numerical rank depends on a tolerance relative to matrix scale and floating-point precision. It is not an exact symbolic statement.

In [59]:
A = np.array([
    [1., 2.],
    [2., 4.],
])

print("rank =", np.linalg.matrix_rank(A))

rank = 1


## 54. Least squares

Least squares finds parameters minimizing a residual norm. A unique numerical solution does not establish model adequacy or experimental identifiability.

$$
\mathbf A\mathbf x\approx\mathbf b
$$

In [60]:
A = np.array([
    [1., 0.],
    [1., 1.],
    [1., 2.],
    [1., 3.],
])

b = np.array([1.0, 2.1, 2.9, 4.2])

x, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

print("x =", x)
print("rank =", rank)
print("singular values =", s)

x = [0.99 1.04]
rank = 2
singular values = [4.10003045 1.09075677]


## 55. Matrix-free operators

A callable can compute Av without storing A. The callable must still encode the correct boundary conditions, units, and linearity.

$$
\mathbf y=\mathcal A(\mathbf x)
$$

In [61]:
def A_operator(x):
    return np.array([
        2*x[0] + x[1],
        x[0] + 3*x[1],
    ])

x = np.array([1., 2.])

print(A_operator(x))

[4. 7.]


## 56. Tractions on many planes

Evaluate several unit normals in a batch. Keep the plane axis separate from the spatial and component axes.

```python
N.shape == (nplane,2)
```

In [62]:
sigma = np.array([
    [12.0, 4.0],
    [4.0, 6.0],
])

angles = np.linspace(0, np.pi, 7)
N = np.column_stack([np.cos(angles), np.sin(angles)])

T = N @ sigma.T

print("N shape =", N.shape)
print("T shape =", T.shape)
print(T)

N shape = (7, 2)
T shape = (7, 2)
[[ 12.           4.        ]
 [ 12.39230485   6.46410162]
 [  9.46410162   7.19615242]
 [  4.           6.        ]
 [ -2.53589838   3.19615242]
 [ -8.39230485  -0.46410162]
 [-12.          -4.        ]]


```python
N @ sigma.T
```

## 57. Normal stress on many planes

Contract both traction and its corresponding normal. A sign-changing normal changes traction but leaves the normal-stress scalar unchanged.

$$
\sigma_n=\mathbf n\cdot\mathbf t
$$

In [63]:
sigma_n = np.einsum("ni,ni->n", N, T)

print(sigma_n)

[12.         13.96410162 10.96410162  6.          4.03589838  7.03589838
 12.        ]


## 58. Batched tensor-vector contraction

Use consistent index labels for sample, output component, and input component. Check against a small explicit loop.

```python
sigma.shape = (N,3,3)
n.shape     = (N,3)
```

In [64]:
Npt = 5

sigma = np.zeros((Npt, 3, 3))
for i in range(Npt):
    sigma[i] = np.diag([1+i, 2+i, 3+i])

n = np.tile(np.array([1., 0., 0.]), (Npt, 1))

t = np.einsum("nij,nj->ni", sigma, n)

print(t)

[[1. 0. 0.]
 [2. 0. 0.]
 [3. 0. 0.]
 [4. 0. 0.]
 [5. 0. 0.]]


## 59. Double contraction of tensor fields

Retain all location axes while summing both component axes. The result is a scalar field rather than a single global scalar.

$$
c(\mathbf x)=\mathbf A(\mathbf x):\mathbf B(\mathbf x)
$$

In [65]:
A = np.ones((4, 5, 3, 3))
B = 2*np.ones((4, 5, 3, 3))

c = np.einsum("...ij,...ij->...", A, B)

print("shape =", c.shape)
print("unique =", np.unique(c))

shape = (4, 5)
unique = [18.]


## 60. Symmetric and skew parts

A splits into one half A plus A-transpose and one half A minus A-transpose. Apply the transpose to component axes only.

$$
\operatorname{sym}\mathbf A=
\frac12(\mathbf A+\mathbf A^T)
$$

$$
\operatorname{skw}\mathbf A=
\frac12(\mathbf A-\mathbf A^T)
$$

In [66]:
A = np.array([
    [1., 3.],
    [2., 4.],
])

symA = 0.5*(A + A.T)
skwA = 0.5*(A - A.T)

print("sym =")
print(symA)
print("skew =")
print(skwA)

sym =
[[1.  2.5]
 [2.5 4. ]]
skew =
[[ 0.   0.5]
 [-0.5  0. ]]


## 61. Deviatoric part

In d dimensions subtract trace(A)/d times the identity. The resulting tensor has zero trace; do not hard-code three in a two-dimensional calculation.

$$
\mathbf A_{\mathrm{dev}}=
\mathbf A-\frac13\operatorname{tr}(\mathbf A)\mathbf I
$$

In [67]:
A = np.diag([6., 3., 0.])

Adev = A - np.trace(A)/3.0 * np.eye(3)

print(Adev)
print("trace dev =", np.trace(Adev))

[[ 3.  0.  0.]
 [ 0.  0.  0.]
 [ 0.  0. -3.]]
trace dev = 0.0


## 62. Component axes and physical meaning

A scalar has no component axis, a vector has one, and a second-order tensor has two. Location and batch axes are additional storage structure.

$$
\boxed{
\text{vector}
\rightarrow
\text{1 component axis}
}
$$

$$
\boxed{
\text{2nd-order tensor}
\rightarrow
\text{2 component axes}
}
$$

## 63. Shape reasoning

A matrix-vector product removes one component index; a double contraction removes two. Write expected shapes before evaluating the expression.

```python
A: (m,n)
x: (n,)
y: (m,)
```

```python
A: (...,d,d)
x: (...,d)
y: (...,d)
```

```python
A: (...,d,d)
B: (...,d,d)
c: (...)
```

## 64. Operation map

Match each mathematical product to star, at, outer, tensordot, or einsum. The index structure determines the appropriate choice.

| Mathematical operation | NumPy |  
| --- | --- |  
| $\mathbf a\cdot\mathbf b$ | `a @ b` |  
| $\|\mathbf a\|$ | `np.linalg.norm(a)` |  
| $\mathbf a\otimes\mathbf b$ | `np.outer(a,b)` |  
| $\mathbf A\mathbf x$ | `A @ x` |  
| $\mathbf A\mathbf B$ | `A @ B` |  
| $\mathbf A^T$ | `A.T` for 2D |  
| trace | `np.trace(A)` |  
| determinant | `np.linalg.det(A)` |  
| solve $Ax=b$ | `np.linalg.solve(A,b)` |  
| eigenproblem | `np.linalg.eigh/eig` |  
| SVD | `np.linalg.svd` |  
| rank | `np.linalg.matrix_rank` |  
| condition | `np.linalg.cond` |  
| double contraction | `np.sum(A*B)` / `einsum` |  
| tensor-vector field contraction | `np.einsum("...ij,...j->...i",A,x)` |  
| tensor component transpose | `np.swapaxes(A,-1,-2)` |

## 65. Continuum mechanics map

Stress maps a plane normal to traction; stiffness maps strain to stress; a projector extracts a component. Each object has a different domain and physical role.

| Physical quantity | Formula | NumPy |  
| --- | --- | --- |  
| traction | $\mathbf t=\boldsymbol\sigma\mathbf n$ | `sigma @ n` |  
| normal stress | $\sigma_n=\mathbf n\cdot\mathbf t$ | `n @ t` |  
| shear stress | $\tau_n=\mathbf m\cdot\mathbf t$ | `m @ t` |  
| strain energy | $\frac12\sigma:\varepsilon$ | `0.5*np.sum(sigma*eps)` |  
| rotated tensor | $\mathbf Q A Q^T$ | `Q @ A @ Q.T` |  
| principal stress | eigenvalues of $\sigma$ | `np.linalg.eigh(sigma)` |

## 66. Transport map

A constitutive tensor maps a gradient to flux. Geometry supplies orientation; measured principal coefficients supply response magnitudes.

| Physical quantity | Formula | NumPy |  
| --- | --- | --- |  
| isotropic flux | $\mathbf q=-D\nabla u$ | `-D * grad_u` |  
| diagonal anisotropic flux | $q_i=-D_i g_i$ | `-Dvec * grad_u` |  
| full tensor flux | $\mathbf q=-\mathbf D\nabla u$ | `-(D @ grad_u)` |  
| rotated diffusivity | $\mathbf D=Q D_m Q^T$ | `Q @ Dmat @ Q.T` |  
| tensor-field flux | $q_i=-D_{ij}g_j$ | `-np.einsum("...ij,...j->...i",D,g)` |

## 67. Three computational categories

Separate elementwise arithmetic, standard linear algebra, and explicit tensor contraction. Choose the operation by meaning rather than convenience.

```python
A * B
```

```python
A @ x
A @ B
```

```python
np.einsum(...)
np.tensordot(...)
```

## 68. When einsum helps

Use einsum when named component contractions clarify a field operation. Readability and a verified index convention matter more than compactness.

```python
A @ x
```

$$
q_i=-D_{ij}g_j
$$

```python
q = -np.einsum("...ij,...j->...i", D, g)
```

## 69. Seven ideas to retain

Distinguish products, normalize directions, declare basis conventions, separate component axes, use structured solvers, assess conditioning, and check invariants.

$$
\boxed{
\texttt{*}=
\text{elementwise multiplication}
}
$$

$$
\boxed{
\texttt{@}=
\text{linear-algebra contraction}
}
$$

## 70. Continue to P5

Fields add spatial location to these numerical objects. P5A introduces location and P5B develops discrete differential operators.

$$
\boxed{
\text{NumPy array}
\rightarrow
\text{discretized physical field}
}
$$

## Supplement: independent numerical checks

These checks target mathematical invariants or an independent reference. They do not validate a wood constitutive law against observations. Tolerances belong to the stated example and tested environment.

In [68]:
# Basis changes must preserve physical flux and quadratic forms.
angle = 0.37
Qcheck = np.array([[np.cos(angle), -np.sin(angle)],
                   [np.sin(angle),  np.cos(angle)]])
Dlocal = np.diag([2e-10, 0.5e-10])
global_gradient = np.array([0.2, -0.7])
local_gradient = Qcheck.T @ global_gradient
flux_global_direct = -(Qcheck @ Dlocal @ Qcheck.T) @ global_gradient
flux_global_via_local = Qcheck @ (-Dlocal @ local_gradient)
np.testing.assert_allclose(flux_global_direct, flux_global_via_local,
                           rtol=1e-13, atol=1e-25)
np.testing.assert_allclose(global_gradient @ (Qcheck @ Dlocal @ Qcheck.T) @ global_gradient,
                           local_gradient @ Dlocal @ local_gradient,
                           rtol=1e-13, atol=1e-25)
assert np.linalg.det(Qcheck) > 0
assert np.min(np.linalg.eigvalsh(Qcheck @ Dlocal @ Qcheck.T)) > 0
# A factor-of-two error at transport scale must not be accepted as 'close'.
assert not scaled_allclose(np.array([1e-10]), np.array([2e-10]))
print("Basis and flux invariance checks passed; transport-scale mismatch rejected.")


Basis and flux invariance checks passed; transport-scale mismatch rejected.


## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [NumPy linear algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.